In [77]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'id_producto': [101, 102, 103, 103, 104, 105, 106, 107, 108, 109, 110, 111],
    'nombre': [' Laptop HP  ', 'Mouse Logitech', 'Teclado Mecanico', 'Teclado Mecanico',
               'Monitor Samsung', None, 'Laptop Dell', 'Auriculares Sony',
               'Tablet Samsung', 'LAPTOP LENOVO', 'Laptop lenovo', 'Webcam Logitech'],
    'categoria': ['Laptop', 'accesorios', 'ACCESORIOS', 'ACCESORIOS', 'Monitores',
                  'laptop', 'Laptop', 'Accesorios', 'Tablets', 'laptop', 'Laptops', 'accesorios'],
    'precio': [1200.50, 25.00, 89.90, 89.90, -50.00, 1500.00, 1350.00,
               45.50, 650.00, 1100.00, 1100.00, 55.00],
    'stock': [15, 120, 45, 45, -10, 8, 22, 80, 35, 12, 12, 95],
    'peso_kg': [2.5, 0.15, 0.8, 0.8, None, None, 2.3, 0.25, 0.55, 2.4, 2.4, 0.18],
    'calificacion': [4.5, 4.2, None, None, 3.8, None, 4.7, 4.0, 4.3, None, None, 3.9],
    'fecha_ingreso': ['2024-05-15', '15/06/2024', '2024-07-20', '2024-07-20',
                      '08-10-2024', None, '2024-09-12', '20/10/2024',
                      '2024-11-05', '11-15-2024', '11-15-2024', '2024-12-01'],
    'pais_origen': ['AR', 'CL', 'CO', 'CO', 'AR', 'CL', 'AR', 'CL', 'CO', 'AR', 'AR', 'CL']
})


In [78]:
registros_iniciales = len(df)
peso_completitud_inicial = df['peso_kg'].notna().mean()*100
completitud_inicial_calif = df['calificacion'].notna().mean()*100


In [79]:
#--------------------------
# 1 Detección de problemas
#--------------------------
print('Valores faltantes: ')
print(df.isnull().sum())
print("\nDuplicados exactos:")
print(df.duplicated().sum())
print("\nDuplicados por ID:")
print(df.duplicated(subset='id_producto').sum())
print("\nPrecios negativos:")
print((df['precio'] < 0).sum())
print("\nStock negativo:")
print((df['stock'] < 0).sum())

#-----------------------------------
# 2 Eliminar registros sin nombre
#-----------------------------------
registros_sin_nombre = df[df['nombre'].isna()]
df= df.dropna(subset=['nombre']).copy()
df= df.reset_index(drop=True)

#------------------------------------
# 3 Normalizar categorías
#------------------------------------
mapa_cat= {
    'Laptop': 'Laptops',
    'laptop': 'Laptops',
    'Laptops': 'Laptops',
    'accesorios': 'Accesorios',
    'ACCESORIOS': 'Accesorios',
    'Accesorios': 'Accesorios',
    'monitores': 'Monitores',
    'Monitores': 'Monitores',
    'tablets': 'Tablets',
    'Tablets': 'Tablets'
}
df['categoria']= df['categoria'].map(mapa_cat)

#-----------------------------------
# 4 Tratamiento de faltantes
#-----------------------------------
# Peso por mediana por categoría
mediana_global_peso = df['peso_kg'].median()
mediana_peso_categoria = (
    df.groupby('categoria')['peso_kg']
      .transform('median')
)
df['peso_kg'] = (
    df['peso_kg']
    .fillna(mediana_peso_categoria)
    .fillna(mediana_global_peso)
)
# Texto "Sin calificación"
df['calificacion'] = (
    df['calificacion']
    .astype('object')
    .fillna('Sin calificación')
)

# Completar fechas usando forward fill
df['fecha_ingreso'] = (
    df['fecha_ingreso'].ffill()
)

#--------------------------------
# 5 Normalizar Nombres
#--------------------------------
df['nombre'] = (
    df['nombre']
    .str.strip()
    .str.title()
)

#--------------------------
# 6 Transformar fechas
#--------------------------
df['fecha_ingreso'] = pd.to_datetime(
    df['fecha_ingreso'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)
df['anio_ingreso'] = (
    df['fecha_ingreso']
    .dt.year
    .astype('Int64')
)
df['mes_ingreso'] = (
    df['fecha_ingreso']
    .dt.month
    .astype('Int64')
)

#----------------------------
# 7 Corregir rangos
#----------------------------
# Reemplaza precios negativos con promedio
precios_validos = (df['precio'].where(df['precio'] >= 0))
prom_precio_categoria = (precios_validos.groupby(df['categoria']).transform('mean'))
prom_precio_global = (precios_validos.mean())
mask_precio_negativo = (df['precio'] < 0)
df.loc[mask_precio_negativo,'precio'] = (prom_precio_categoria[mask_precio_negativo].fillna(prom_precio_global))

# Reemplaza stock negativo con 0
df.loc[df["stock"] < 0, "stock"] = 0

#---------------------------------
# 8 Eliminar duplicados y validar
#---------------------------------
registros_duplicados = len(df)
df = (df.drop_duplicates(subset=['id_producto'], keep='first').copy().reset_index(drop=True))
duplicados_eliminados = (registros_duplicados - len(df))
categorias_validas = {
    'Laptops',
    'Accesorios',
    'Monitores',
    'Tablets'
}

ids_unicos = df["id_producto"].is_unique
categorias_validas = set(df["categoria"]).issubset(
    {"Laptops", "Accesorios", "Monitores", "Tablets"}
)

precio_valido = (df["precio"] >= 0).all()
stock_valido = (df["stock"] >= 0).all()

registros_finales = len(df)
peso_completitud_final = df["peso_kg"].notna().mean() * 100
calif_completitud_final = (
    df["calificacion"] != "Sin calificación"
).mean() * 100

#----------------------
# Reporte final
#----------------------

print("\n========== REPORTE FINAL ==========")
print(f"Registros iniciales: {registros_iniciales}")
print(f"Registros finales: {registros_finales}")
print(
    f"\nCompletitud peso_kg: "
    f"{peso_completitud_inicial:.1f}% → {peso_completitud_final:.1f}%"
)
print(
    f"Completitud calificación: "
    f"{completitud_inicial_calif:.1f}% → {calif_completitud_final:.1f}%"
)
print(f"\nDuplicados eliminados: {duplicados_eliminados}")
print("\nValidaciones")
print("IDs únicos:", ids_unicos)
print("Categorías válidas:", categorias_validas)
print("Precios válidos:", precio_valido)
print("Stock válido:", stock_valido)
if all([ids_unicos, categorias_validas, precio_valido, stock_valido]):
    print("\n Todas las validaciones se realizaron exitosamente.")
else:
    print("\n Existen validaciones que requieren revisión.")


#---------------
# Exportar a csv
#---------------
df.to_csv(
    "archivo_limpio.csv",
    index=False,
    encoding="utf-8-sig"
)




Valores faltantes: 
id_producto      0
nombre           1
categoria        0
precio           0
stock            0
peso_kg          2
calificacion     5
fecha_ingreso    1
pais_origen      0
dtype: int64

Duplicados exactos:
1

Duplicados por ID:
1

Precios negativos:
1

Stock negativo:
1

========== REPORTE FINAL ==========
Registros iniciales: 12
Registros finales: 10

Completitud peso_kg: 83.3% → 100.0%
Completitud calificación: 58.3% → 70.0%

Duplicados eliminados: 1

Validaciones
IDs únicos: True
Categorías válidas: True
Precios válidos: True
Stock válido: True

 Todas las validaciones se realizaron exitosamente.
